# 🐶 Dog Breed Classification using Transfer Learning

This notebook demonstrates how to build a dog breed classifier using **Transfer Learning** with InceptionV3 pre-trained on ImageNet.

## Project Overview
- **Dataset**: 10,000 images across 120 dog breeds
- **Model**: InceptionV3 (pre-trained on ImageNet) + custom classification head
- **Technique**: Transfer Learning with fine-tuning
- **Framework**: TensorFlow / Keras

## Dataset
Download the dataset from: [Dog Breed Identification Dataset](https://drive.google.com/file/d/14of-v7y9Q95fqBvOXfTrEg2C2LZ-aq6l/view?usp=sharing)

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import cv2
import tensorflow as tf
from tensorflow import keras
from keras import layers
from functools import partial

import warnings
warnings.filterwarnings('ignore')

AUTO = tf.data.experimental.AUTOTUNE
print('TensorFlow version:', tf.__version__)

## Step 2: Load Dataset

In [ ]:
from zipfile import ZipFile

data_path = 'dog-breed-identification.zip'

with ZipFile(data_path, 'r') as zip:
    zip.extractall()
    print('Dataset extracted successfully.')

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
df = pd.read_csv('labels.csv')
print('Dataset shape:', df.shape)
df.head()

In [ ]:
print('Number of unique dog breeds:', df['breed'].nunique())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(14, 5))
df['breed'].value_counts().plot.bar()
plt.title('Number of Images per Dog Breed')
plt.xlabel('Breed')
plt.ylabel('Count')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create full filepath column
df['filepath'] = 'train/' + df['id'] + '.jpg'
df.head()

In [ ]:
# Visualize sample images
plt.subplots(figsize=(12, 12))
for i in range(12):
    plt.subplot(4, 3, i+1)
    k = np.random.randint(0, len(df))
    img = cv2.imread(df.loc[k, 'filepath'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.title(df.loc[k, 'breed'], fontsize=9)
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Encode labels
le = LabelEncoder()
df['breed'] = le.fit_transform(df['breed'])
print('Label encoding complete. Sample:')
df.head()

## Step 4: Data Preparation — Train/Validation Split

In [ ]:
features = df['filepath']
target = df['breed']

X_train, X_val, Y_train, Y_val = train_test_split(
    features, target,
    test_size=0.15,
    random_state=10
)

print(f'Training samples:   {X_train.shape[0]}')
print(f'Validation samples: {X_val.shape[0]}')

## Step 5: Image Augmentation with Albumentations

In [ ]:
import albumentations as A

transforms_train = A.Compose([
    A.VerticalFlip(p=0.2),
    A.HorizontalFlip(p=0.7),
    A.CoarseDropout(p=0.5),
    A.RandomGamma(p=0.5),
    A.RandomBrightnessContrast(p=1)
])

In [ ]:
# Visualize augmentations on a sample image
sample_path = df[df['filepath'].str.startswith('train/')].iloc[0]['filepath']
img = cv2.imread(sample_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

augments = [
    ('Original', lambda x: x),
    ('Horizontal Flip', A.HorizontalFlip(p=1)),
    ('Vertical Flip', A.VerticalFlip(p=1)),
    ('Coarse Dropout', A.CoarseDropout(p=1)),
    ('CLAHE', A.CLAHE(p=1)),
    ('Brightness/Contrast', A.RandomBrightnessContrast(p=1))
]

plt.figure(figsize=(14, 5))
for i, (name, aug) in enumerate(augments):
    plt.subplot(2, 3, i+1)
    if callable(aug) and not hasattr(aug, '__call__'):
        aug_img = img
    else:
        try:
            aug_img = aug(image=img)['image']
        except:
            aug_img = img
    plt.imshow(aug_img)
    plt.title(name)
    plt.axis('off')
plt.suptitle('Data Augmentation Examples', fontsize=13)
plt.tight_layout()
plt.show()

## Step 6: Build Input Pipeline (tf.data)

In [ ]:
def aug_fn(img):
    aug_data = transforms_train(image=img)
    return aug_data['image']


@tf.function
def process_data(img, label):
    aug_img = tf.numpy_function(aug_fn, [img], Tout=tf.float32)
    return img, label


def decode_image(filepath, label=None):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img)
    img = tf.image.resize(img, [128, 128])
    img = tf.cast(img, tf.float32) / 255.0

    if label is None:
        return img
    return img, tf.one_hot(indices=label, depth=120, dtype=tf.float32)

In [ ]:
BATCH_SIZE = 32

train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train, Y_train))
    .map(decode_image, num_parallel_calls=AUTO)
    .map(partial(process_data), num_parallel_calls=AUTO)
    .batch(BATCH_SIZE)
    .prefetch(AUTO)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_val, Y_val))
    .map(decode_image, num_parallel_calls=AUTO)
    .batch(BATCH_SIZE)
    .prefetch(AUTO)
)

# Verify shapes
for img, label in train_ds.take(1):
    print('Image batch shape:', img.shape)
    print('Label batch shape:', label.shape)

## Step 7: Build the Model using Transfer Learning (InceptionV3)

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3

# Load pre-trained InceptionV3 (without top classification layers)
pre_trained_model = InceptionV3(
    input_shape=(128, 128, 3),
    weights='imagenet',
    include_top=False
)

print(f'Total layers in InceptionV3: {len(pre_trained_model.layers)}')

In [ ]:
# Freeze all pre-trained layers
for layer in pre_trained_model.layers:
    layer.trainable = False

# Use output from 'mixed7' layer as feature extractor
last_layer = pre_trained_model.get_layer('mixed7')
print('Feature map shape:', last_layer.output.shape)
last_output = last_layer.output

In [ ]:
# Build custom classification head on top of InceptionV3
x = layers.Flatten()(last_output)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.BatchNormalization()(x)
output = layers.Dense(120, activation='softmax')(x)

model = keras.Model(pre_trained_model.input, output)

# Compile the model
model.compile(
    optimizer='adam',
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.AUC()]
)

model.summary()

## Step 8: Train the Model

In [ ]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau


class AUCThresholdCallback(tf.keras.callbacks.Callback):
    """Stop training when validation AUC exceeds 0.99."""
    def on_epoch_end(self, epoch, logs={}):
        val_auc = logs.get('val_auc')
        if val_auc is not None and val_auc > 0.99:
            print(f'\nVal AUC reached {val_auc:.4f}. Stopping training.')
            self.model.stop_training = True


callbacks = [
    EarlyStopping(
        patience=3,
        monitor='val_auc',
        restore_best_weights=True,
        mode='max'
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        patience=2,
        factor=0.5,
        verbose=1
    ),
    AUCThresholdCallback()
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    verbose=1,
    callbacks=callbacks
)

## Step 9: Evaluate and Visualize Results

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history_df['loss'], label='Train Loss')
axes[0].plot(history_df['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# AUC plot
axes[1].plot(history_df['auc'], label='Train AUC')
axes[1].plot(history_df['val_auc'], label='Val AUC')
axes[1].set_title('AUC over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Training History', fontsize=14)
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best Val AUC: {history_df['val_auc'].max():.4f}")
print(f"Best Val Loss: {history_df['val_loss'].min():.4f}")

In [ ]:
# Save the trained model
model.save('dog_breed_model.h5')
print('Model saved as dog_breed_model.h5')

## Step 10: Predict on a Sample Image

In [ ]:
def predict_breed(image_path, model, label_encoder):
    """Predict the breed of a dog from an image path."""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    img_resized = cv2.resize(img_rgb, (128, 128))
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    predictions = model.predict(img_batch, verbose=0)
    top_indices = np.argsort(predictions[0])[::-1][:3]
    
    print('Top 3 Predictions:')
    for i, idx in enumerate(top_indices):
        breed_name = label_encoder.inverse_transform([idx])[0]
        confidence = predictions[0][idx] * 100
        print(f'  {i+1}. {breed_name}: {confidence:.2f}%')
    
    plt.imshow(img_rgb)
    plt.title(f"Predicted: {label_encoder.inverse_transform([top_indices[0]])[0]}")
    plt.axis('off')
    plt.show()


# Example usage (replace with any image path)
sample_image = df[df['filepath'].str.startswith('train/')].sample(1).iloc[0]
original_breed = le.inverse_transform([sample_image['breed']])[0]
print(f'Actual breed: {original_breed}')
predict_breed(sample_image['filepath'], model, le)